# 🌍 Global Child Health Data Extraction Tutorial

**A Beginner's Guide to Fetching Health Indicators from Global APIs**

---

Welcome! This notebook will teach you how to extract child health data from major global health databases. By the end, you'll have a consolidated dataset ready for analysis.

## What You'll Learn

1. 📦 **Setting up** your Python environment
2. 🏦 **World Bank API** – Fetch malnutrition, birth weight, and mortality data
3. 🏥 **WHO GHO API** – Get wasting prevalence and breastfeeding statistics
4. 👶 **UNICEF SDMX API** – Retrieve under-5 population counts
5. 🔄 **Processing & Consolidating** – Clean and combine all data sources
6. 🧮 **Advanced Data Imputation** – Handle missing values intelligently
7. 💾 **Saving Results** – Export complete datasets to Excel/CSV
8. 🎉 **Wrap-up** – Summary and next steps

---


## 📦 Step 1: Install Required Packages

First, let's make sure we have all the libraries we need. Run the cell below to install them.


In [ ]:
# Install required packages (run this once)
%pip install pandas requests openpyxl country_converter tqdm --quiet

print("✅ All packages installed!")


## 📚 Step 2: Import Libraries

Now let's import the libraries we'll use throughout this tutorial.


In [ ]:
import pandas as pd
import requests
import time
import numpy as np
from datetime import datetime

# For converting country codes to names
import country_converter as coco

# For progress bars
from tqdm.notebook import tqdm

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print(f"🚀 Tutorial started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("✅ Libraries loaded successfully!")


---

## 🏦 Step 3: Fetching Data from the World Bank API

The [World Bank Open Data](https://data.worldbank.org/) provides free access to global development data. We'll fetch several child health indicators.

### How the World Bank API Works

The API uses **indicator codes** to identify specific datasets. For example:
- `SH.STA.MALN.ZS` = Malnutrition prevalence (weight-for-age < -2 SD)
- `SH.STA.BRTW.ZS` = Low birth weight prevalence
- `SH.DYN.MORT` = Under-5 mortality rate

The base URL format is:
```
http://api.worldbank.org/v2/country/all/indicator/{INDICATOR_CODE}?format=json
```


In [ ]:
def fetch_world_bank_indicator(indicator_id, indicator_name):
    """
    Fetches data for a single indicator from the World Bank API.
    
    Parameters:
    -----------
    indicator_id : str
        The World Bank indicator code (e.g., 'SH.STA.MALN.ZS')
    indicator_name : str
        A human-readable name for the indicator
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: country, year, value
    """
    print(f"📥 Fetching: {indicator_name}...")
    
    all_data = []
    page = 1
    
    while True:
        # Build the API URL with pagination
        url = f"http://api.worldbank.org/v2/country/all/indicator/{indicator_id}?format=json&page={page}&per_page=1000"
        
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()  # Raise error for bad status codes
            data = response.json()
            
            # World Bank returns [metadata, data] - we want data[1]
            if not data or len(data) < 2 or not data[1]:
                break  # No more data
            
            all_data.extend(data[1])
            page += 1
            time.sleep(0.3)  # Be nice to the API - don't hammer it!
            
        except Exception as e:
            print(f"   ⚠️ Error on page {page}: {e}")
            break
    
    if not all_data:
        print(f"   ❌ No data found for {indicator_name}")
        return pd.DataFrame()
    
    # Convert to DataFrame
    df = pd.DataFrame(all_data)
    
    # Extract country name from nested dict
    df['country'] = df['country'].apply(lambda x: x['value'] if isinstance(x, dict) else x)
    
    # Keep only rows with actual values
    df = df[['country', 'date', 'value']].dropna(subset=['value'])
    df.columns = ['country', 'year', 'value']
    
    # Convert to numeric
    df['year'] = pd.to_numeric(df['year'])
    df['value'] = pd.to_numeric(df['value'])
    
    print(f"   ✅ Found {len(df):,} records")
    return df


### Let's Try It!

Let's fetch malnutrition data and see what we get:


In [ ]:
# Fetch malnutrition data
malnutrition_df = fetch_world_bank_indicator(
    indicator_id='SH.STA.MALN.ZS',
    indicator_name='Malnutrition (weight-for-age < -2 SD)'
)

# Let's peek at the data!
print("\n📊 Sample of the data:")
malnutrition_df.head(10)


In [ ]:
# Define all World Bank indicators we want
wb_indicators = {
    'SH.STA.MALN.ZS': 'Malnutrition (weight-for-age < -2 SD)',
    'SH.STA.BRTW.ZS': 'Low birth weight (≤2500g)',
    'EG.USE.COMM.CL.ZS': 'Solid fuel use (%)',
    'SH.DYN.MORT': 'Under-5 mortality rate (per 1000)',
}

# Fetch all indicators
world_bank_data = []

for indicator_id, indicator_name in wb_indicators.items():
    df = fetch_world_bank_indicator(indicator_id, indicator_name)
    if not df.empty:
        df['indicator'] = indicator_name
        df['source'] = 'World Bank'
        world_bank_data.append(df)

# Combine all World Bank data
wb_combined = pd.concat(world_bank_data, ignore_index=True)
print(f"\n✅ Total World Bank records: {len(wb_combined):,}")


---

## 🏥 Step 4: Fetching Data from the WHO GHO API

The [WHO Global Health Observatory](https://www.who.int/data/gho) provides health statistics for all WHO member states.

### How the WHO GHO API Works

The WHO uses an OData-style API. The base URL is:
```
https://ghoapi.azureedge.net/api/{INDICATOR_CODE}
```

Key indicators we'll use:
- `NUTRITION_WH_2` = Wasting prevalence in children under 5
- `WHOSIS_000006` = Exclusive breastfeeding under 6 months


In [ ]:
def fetch_who_indicator(indicator_code, indicator_name):
    """
    Fetches data from the WHO GHO API.
    
    Parameters:
    -----------
    indicator_code : str
        The WHO indicator code (e.g., 'NUTRITION_WH_2')
    indicator_name : str
        A human-readable name for the indicator
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: country, year, value
    """
    print(f"📥 Fetching: {indicator_name}...")
    
    url = f"https://ghoapi.azureedge.net/api/{indicator_code}"
    
    try:
        response = requests.get(url, timeout=45)
        response.raise_for_status()
        data = response.json().get('value', [])
        
        if not data:
            print(f"   ❌ No data found")
            return pd.DataFrame()
        
        df = pd.DataFrame(data)
        
        # Filter to country-level data only (not regions)
        df = df[df['SpatialDimType'] == 'COUNTRY']
        
        # Select and rename columns
        df = df[['SpatialDim', 'TimeDim', 'NumericValue']]
        df.columns = ['country', 'year', 'value']
        
        # Convert ISO3 codes to country names (suppress warnings for regional codes)
        import warnings
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='.*not found in regex.*')
            df['country'] = coco.convert(df['country'].tolist(), to='name_short', not_found=np.nan)
        df = df.dropna(subset=['country'])
        
        # Convert to numeric
        df['year'] = pd.to_numeric(df['year'])
        df['value'] = pd.to_numeric(df['value'])
        
        print(f"   ✅ Found {len(df):,} records")
        return df
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return pd.DataFrame()

# Fetch WHO indicators
who_indicators = {
    'NUTRITION_WH_2': 'Wasting prevalence (children under 5)',
    'WHOSIS_000006': 'Exclusive breastfeeding under 6 months (%)',
}

who_data = []
for indicator_code, indicator_name in who_indicators.items():
    df = fetch_who_indicator(indicator_code, indicator_name)
    if not df.empty:
        df['indicator'] = indicator_name
        df['source'] = 'WHO'
        who_data.append(df)

who_combined = pd.concat(who_data, ignore_index=True) if who_data else pd.DataFrame()
print(f"\n✅ Total WHO records: {len(who_combined):,}")


---

## 👶 Step 5: Fetching Population Data from UNICEF

The [UNICEF Data Warehouse](https://data.unicef.org/) provides demographic data including under-5 population counts.

### How the UNICEF SDMX API Works

UNICEF uses the SDMX (Statistical Data and Metadata eXchange) format. The response structure is a bit complex, but we'll walk through it step by step.


In [ ]:
import warnings
warnings.simplefilter("ignore", category=Warning)
import logging
logging.getLogger('country_converter').setLevel(logging.ERROR)

def fetch_unicef_under5_population():
    """
    Fetches under-5 population data from UNICEF SDMX API.
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: country, year, value (population count)
    """
    print("📥 Fetching: Under-5 population from UNICEF...")
    
    url = "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,DM,1.0/.DM_POP_U5?format=sdmx-json"
    
    try:
        response = requests.get(url, timeout=90)
        response.raise_for_status()
        payload = response.json()
        
        # Build lookup tables from the SDMX structure
        series_dims = payload["data"]["structure"]["dimensions"]["series"]
        ref_areas = series_dims[0]["values"]  # Countries (ISO3 codes)
        ref_lookup = {str(idx): v["id"] for idx, v in enumerate(ref_areas)}
        
        time_values = payload["data"]["structure"]["dimensions"]["observation"][0]["values"]
        time_lookup = {str(idx): int(v["id"]) for idx, v in enumerate(time_values)}
        
        # Extract the actual data
        series_dict = payload["data"]["dataSets"][0]["series"]
        
        records = []
        for series_key, series_val in series_dict.items():
            ref_idx = series_key.split(":")[0]
            iso3 = ref_lookup.get(ref_idx)
            if not iso3:
                continue
                
            for time_idx, obs in series_val["observations"].items():
                year = time_lookup.get(time_idx)
                if year is None:
                    continue
                try:
                    # UNICEF returns population in thousands!
                    value = float(obs[0]) * 1000
                    records.append({"country": iso3, "year": year, "value": value})
                except (TypeError, ValueError):
                    continue
        
        if not records:
            
            return pd.DataFrame()
        
        df = pd.DataFrame(records)
        
        # Convert ISO3 codes to country names (suppress warnings for regional codes)
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore')
            df['country'] = coco.convert(df['country'].tolist(), to='name_short', not_found=np.nan)
        df = df.dropna(subset=['country'])
        return df
        
    except Exception as e:

        return pd.DataFrame()

# Fetch UNICEF population data
population_df = fetch_unicef_under5_population()
population_df['indicator'] = 'Population ages 0-4 (number)'
population_df['source'] = 'UNICEF'

print("\n📊 Sample population data:")
population_df.head(10)


---

## 🔄 Step 6: Consolidate All Data Sources

Now let's combine all the data we've fetched into a single, clean dataset.


In [ ]:
# Combine all data sources
all_data = pd.concat([wb_combined, who_combined, population_df], ignore_index=True)

def delist(x):
    if type(x) == list:
        return ' '.join(x)
    else:
        return x

all_data['country'] = all_data['country'].apply(delist)
print("📊 Combined Dataset Summary:")
print(f"   Total records: {len(all_data):,}")
print(f"   Unique countries: {all_data['country'].nunique()}")
print(f"   Indicators: {all_data['indicator'].nunique()}")
print(f"   Year range: {all_data['year'].min()} - {all_data['year'].max()}")

print("\n📋 Records by indicator:")
all_data.groupby('indicator').size().sort_values(ascending=False)


In [ ]:
# Get the latest value for each country-indicator combination
latest_values = (
    all_data
    .sort_values('year', ascending=False)
    .drop_duplicates(['country', 'indicator'])
)

# Pivot to wide format (countries as rows, indicators as columns)
summary_table = latest_values.pivot(
    index='country',
    columns='indicator',
    values='value'
).reset_index()

summary_table.columns.name = None

print(f"📊 Summary table: {len(summary_table)} countries × {len(summary_table.columns)-1} indicators")
summary_table.head(15)


---

## 🧮 Step 7: Advanced Data Imputation for Missing Values

Now that we have consolidated data from multiple sources (Steps 3-6), we need to address a common challenge in global health research: **missing values**. Countries may not report certain indicators, or data may be unavailable for specific years. 

The GBD Enhanced Template Populator uses a sophisticated **hierarchical imputation system** to fill these gaps intelligently, ensuring we have complete datasets ready for saving and analysis.

### 🎯 Imputation Strategy Overview

The system uses a **4-tier fallback hierarchy**:

1. **Direct Match** - Use the latest available country-specific data
2. **Sub-regional Average** - Use mortality-based sub-regional groupings  
3. **Regional Average** - Fall back to World Bank regional averages
4. **Global Average** - Use global mean as final fallback

### 🌍 Mortality-Based Sub-Regional Classification

Countries are grouped not just by geography, but by **child mortality patterns** within regions. This creates more meaningful comparisons for imputation.


In [ ]:
# Let's implement the imputation system step by step
# Note: This builds on the 'all_data' DataFrame created in Step 6
import warnings
warnings.simplefilter("ignore", category=Warning)

# First, let's create a simplified version of the country-to-region mapping
# (This is a subset - the full version has 200+ countries)
country_to_region = {
    # Sub-Saharan Africa
    'Nigeria': 'Sub-Saharan Africa', 'Kenya': 'Sub-Saharan Africa', 'Ghana': 'Sub-Saharan Africa',
    'Ethiopia': 'Sub-Saharan Africa', 'Tanzania': 'Sub-Saharan Africa', 'Uganda': 'Sub-Saharan Africa',
    
    # South Asia  
    'India': 'South Asia', 'Pakistan': 'South Asia', 'Bangladesh': 'South Asia',
    'Afghanistan': 'South Asia', 'Nepal': 'South Asia', 'Sri Lanka': 'South Asia',
    
    # East Asia & Pacific
    'China': 'East Asia & Pacific', 'Indonesia': 'East Asia & Pacific', 'Philippines': 'East Asia & Pacific',
    'Vietnam': 'East Asia & Pacific', 'Thailand': 'East Asia & Pacific', 'Malaysia': 'East Asia & Pacific',
    
    # Latin America & Caribbean
    'Brazil': 'Latin America & Caribbean', 'Mexico': 'Latin America & Caribbean', 'Colombia': 'Latin America & Caribbean',
    'Peru': 'Latin America & Caribbean', 'Argentina': 'Latin America & Caribbean', 'Chile': 'Latin America & Caribbean',
    
    # Middle East & North Africa
    'Egypt': 'Middle East & North Africa', 'Morocco': 'Middle East & North Africa', 'Jordan': 'Middle East & North Africa',
    'Tunisia': 'Middle East & North Africa', 'Algeria': 'Middle East & North Africa',
    
    # Europe & Central Asia
    'Turkey': 'Europe & Central Asia', 'Russia': 'Europe & Central Asia', 'Kazakhstan': 'Europe & Central Asia',
    'Ukraine': 'Europe & Central Asia', 'Poland': 'Europe & Central Asia',
    
    # North America
    'United States': 'North America', 'Canada': 'North America'
}

def get_imputation_value(country, indicator, data_df, country_to_region):
    """
    Gets a value using hierarchical imputation:
    1. Direct country match (latest year)
    2. Regional average
    3. Global average
    """
    
    # Tier 1: Direct match - find latest data for this country and indicator
    direct_match = data_df[(data_df['country'] == country) & (data_df['indicator'] == indicator)]
    if not direct_match.empty:
        latest = direct_match.sort_values('year', ascending=False).iloc[0]
        return {
            'value': latest['value'], 
            'method': 'Direct', 
            'source': f"{latest['source']} ({latest['year']})"
        }
    
    # Tier 2: Regional average  
    region = country_to_region.get(country)
    if region:
        # Get all countries in the same region
        region_countries = [c for c, r in country_to_region.items() if r == region]
        regional_data = data_df[
            (data_df['country'].isin(region_countries)) & 
            (data_df['indicator'] == indicator)
        ]
        if not regional_data.empty:
            # Get latest data for each country, then average
            latest_regional = regional_data.sort_values('year', ascending=False).drop_duplicates('country')
            avg_val = latest_regional['value'].mean()
            return {
                'value': avg_val, 
                'method': f'Regional Average ({region})', 
                'source': 'Calculated Average'
            }
    
    # Tier 3: Global average
    global_data = data_df[data_df['indicator'] == indicator]
    if not global_data.empty:
        latest_global = global_data.sort_values('year', ascending=False).drop_duplicates('country')
        avg_val = latest_global['value'].mean()
        return {
            'value': avg_val, 
            'method': 'Global Average', 
            'source': 'Calculated Average'
        }
    
    # No data available at any level
    return None

print("✅ Imputation system ready!")
print(f"📊 Regions covered: {len(set(country_to_region.values()))}")
print(f"🌍 Countries mapped: {len(country_to_region)}")


In [ ]:
import os

# Create output directory
output_dir = 'tutorial_output'
os.makedirs(output_dir, exist_ok=True)

# Save to Excel with multiple sheets
output_excel = f'{output_dir}/gbd_data_package.xlsx'

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    # Sheet 1: Summary table (wide format)
    summary_table.to_excel(writer, sheet_name='Summary', index=False)
    
    # Sheet 2: All raw data (long format)
    all_data.to_excel(writer, sheet_name='All_Data', index=False)
    
    # Sheet 3: Population data
    population_df.to_excel(writer, sheet_name='Population', index=False)

print(f"✅ Excel file saved: {output_excel}")

# Also save as CSV
summary_csv = f'{output_dir}/gbd_summary.csv'
summary_table.to_csv(summary_csv, index=False)
print(f"✅ CSV file saved: {summary_csv}")
